# Feature-Stacking Ensemble (Model 3)

Обучает третью модель поверх **декодерных признаков** двух замороженных базовых моделей:
- **Model 1**: `UnetPlusPlus` + `efficientnet-b3` (512×512)
- **Model 2**: `DeepLabV3Plus` + `tu-convnext_base` (352→512)

Архитектура стекинга:
```
Image → [Encoder1 → Decoder1] → feats1 (16 ch)  ─┐
                                                    ├─ cat(272 ch) → MetaHead → logits
Image → [Encoder2 → Decoder2] → feats2 (256 ch) ─┘
```
Базовые модели **заморожены**, обучается только `MetaHead`.

## 0. rclone setup (Kaggle Secrets)

In [20]:
!curl https://rclone.org/install.sh --output install.sh

# Даем права на выполнение и запускаем
!chmod +x install.sh
!./install.sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4734  100  4734    0     0  11692      0 --:--:-- --:--:-- --:--:-- 11717

The latest version of rclone rclone v1.73.4 is already installed.



In [21]:
import os, re, subprocess
from pathlib import Path
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
rclone_conf = secrets.get_secret("RCLONE_CONF")

# Восстанавливаем переносы строк (Kaggle Secrets схлопывает их)
rclone_conf = re.sub(
    r'\s+(type|scope|token|team_drive|root_folder_id)\s*=',
    r'\n\1 =',
    rclone_conf
)

conf_path = Path("/root/.config/rclone/rclone.conf")
conf_path.parent.mkdir(parents=True, exist_ok=True)
conf_path.write_text(rclone_conf)

result = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
print("Remotes:", result.stdout)
print("rclone configured ✓")

Remotes: gdrive:

rclone configured ✓


## 1. Install dependencies

In [22]:
!pip install --upgrade pip
!pip install -q segmentation-models-pytorch timm albumentations scikit-learn

^C
ERROR: Operation cancelled by user


## 2. GPU optimizations

In [23]:
import torch

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.enabled   = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

print("CUDA optimizations enabled")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CUDA optimizations enabled
GPU: Tesla T4


## 3. Config

In [24]:
from pathlib import Path
import torch

# ── Пути к данным ────────────────────────────────────────────
DATA_ROOT  = Path("/kaggle/input/competitions/dl-lab-3-product-segmentation/train")
IMAGES_DIR = DATA_ROOT / "images"
MASKS_DIR  = DATA_ROOT / "masks"

SAVE_DIR = Path("/kaggle/working/stacked_runs")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Google Drive (rclone) ────────────────────────────────────
GDRIVE_REMOTE    = "gdrive"
GDRIVE_SAVE_PATH = "feature_stacking"  # папка с model1 и model2

CKPT1_REMOTE = f"{GDRIVE_REMOTE}:{GDRIVE_SAVE_PATH}/model_1/best(2).pth"
CKPT2_REMOTE = f"{GDRIVE_REMOTE}:{GDRIVE_SAVE_PATH}/model_2/best(1).pth"

# ── Конфиги базовых моделей ───────────────────────────────────
MODEL1_CONFIG = dict(
    model_name      = "UnetPlusPlus",
    encoder_name    = "efficientnet-b3",
    encoder_weights = "imagenet",
)
MODEL2_CONFIG = dict(
    model_name      = "DeepLabV3Plus",
    encoder_name    = "tu-convnext_base",
    encoder_weights = "imagenet",
)

# ── Гиперпараметры стекинга ───────────────────────────────────
IMG_SIZE = 352  # Оптимальный размер
BATCH_SIZE = 24  # Подходит для EfficientNetV2-S на T4/V100
NUM_EPOCHS   = 30
LR           = 1e-4
WEIGHT_DECAY = 1e-4
VAL_RATIO    = 0.15

NUM_WORKERS        = 2
PREFETCH_FACTOR    = 2
PERSISTENT_WORKERS = True

SEED      = 42
THRESHOLD = 0.5
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device : {DEVICE}")
print(f"Data   : {DATA_ROOT}")

Device : cuda
Data   : /kaggle/input/competitions/dl-lab-3-product-segmentation/train


## 4. Download checkpoints from Drive

In [25]:
import subprocess

def download_from_drive(remote_path: str, local_path: Path) -> Path:
    """Скачивает файл с Google Drive через rclone."""
    result = subprocess.run(
        ["rclone", "copyto", remote_path, str(local_path)],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise FileNotFoundError(
            f"rclone failed for {remote_path}:\n{result.stderr.strip()}"
        )
    print(f"✓ Downloaded: {remote_path} → {local_path}")
    return local_path


ckpt1_local = download_from_drive(CKPT1_REMOTE, SAVE_DIR / "base_model1.pth")
ckpt2_local = download_from_drive(CKPT2_REMOTE, SAVE_DIR / "base_model2.pth")

KeyboardInterrupt: 

## 5. Imports

In [ ]:
import random
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn
import albumentations as A
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import sys

## 6. Utils

In [ ]:
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def find_image_for_stem(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None


def dice_score_from_logits(
    logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7
) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()
    preds   = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (preds * targets).sum(dim=1)
    denom = preds.sum(dim=1) + targets.sum(dim=1)
    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean().item()


def iou_score_from_logits(
    logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-7
) -> float:
    probs = torch.sigmoid(logits)
    preds = (probs > THRESHOLD).float()
    preds   = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)
    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()


seed_everything(SEED)

## 7. Loss

In [ ]:
bce_loss  = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode=smp.losses.BINARY_MODE, from_logits=True)

def combined_loss(logits, targets):
    return 0.25 * bce_loss(logits, targets) + 0.75 * dice_loss(logits, targets)

## 8. Augmentations

In [ ]:
def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    ], p=1.0)

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
    ], p=1.0)

## 9. Dataset

Ключевое отличие от обычного датасета: одно изображение проходит **два разных preprocessing** (по одному на каждый энкодер) — потому что нормализация ImageNet у `efficientnet-b3` и `tu-convnext_base` разная.

Возвращает `(img1, img2, mask)` — оба тензора нормализованы под свой энкодер.

In [ ]:
class StackedDataset(Dataset):
    """
    Датасет для feature-stacking ансамбля.

    Возвращает:
        img1  : тензор [3, H, W], нормализованный под encoder1
        img2  : тензор [3, H, W], нормализованный под encoder2
        mask  : тензор [1, H, W], бинарная маска {0, 1}
    """

    def __init__(
        self,
        images_dir: Path,
        masks_dir: Path,
        img_size: int,
        enc1_name: str,
        enc1_weights: str | None,
        enc2_name: str,
        enc2_weights: str | None,
        transforms=None,
    ):
        self.images_dir = Path(images_dir)
        self.masks_dir  = Path(masks_dir)
        self.img_size   = img_size
        self.transforms = transforms

        # Каждый энкодер имеет свою нормализацию (mean/std)
        self.preprocess1 = (
            get_preprocessing_fn(enc1_name, pretrained=enc1_weights)
            if enc1_weights else None
        )
        self.preprocess2 = (
            get_preprocessing_fn(enc2_name, pretrained=enc2_weights)
            if enc2_weights else None
        )

        self.samples = []
        for mask_path in sorted(self.masks_dir.glob("*.png")):
            stem = mask_path.stem
            image_path = find_image_for_stem(self.images_dir, stem)
            if image_path is not None:
                self.samples.append((image_path, mask_path))

        if not self.samples:
            raise RuntimeError(
                f"No paired samples found in {self.images_dir} & {self.masks_dir}"
            )
        print(f"Found {len(self.samples)} paired samples")

    def __len__(self) -> int:
        return len(self.samples)

    def _to_tensor(self, image_rgb: np.ndarray, preprocess_fn) -> torch.Tensor:
        """float32 HWC numpy → нормализованный CHW тензор."""
        img = image_rgb.astype(np.float32)
        if preprocess_fn is not None:
            img = preprocess_fn(img)
        else:
            img = img / 255.0
        return torch.from_numpy(img.transpose(2, 0, 1)).float()

    def __getitem__(self, idx: int):
        image_path, mask_path = self.samples[idx]

        # Читаем сырые данные
        image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        mask      = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        # Аугментации — применяем один раз к сырому RGB
        if self.transforms is not None:
            augmented = self.transforms(image=image_rgb, mask=mask)
            image_rgb = augmented["image"]
            mask      = augmented["mask"]
        else:
            image_rgb = cv2.resize(image_rgb, (self.img_size, self.img_size))
            mask      = cv2.resize(
                mask, (self.img_size, self.img_size),
                interpolation=cv2.INTER_NEAREST
            )

        # Применяем preprocessing каждого энкодера отдельно
        img1 = self._to_tensor(image_rgb, self.preprocess1)
        img2 = self._to_tensor(image_rgb, self.preprocess2)

        mask = (mask > 0).astype(np.float32)
        mask = torch.from_numpy(mask[None, ...]).float()

        return img1, img2, mask

## 10. Feature-Stacking Model

Архитектура:
- `model1.encoder` + `model1.decoder` → `feats1` ([B, 16, H, W] для UnetPlusPlus/efficientnet-b3)
- `model2.encoder` + `model2.decoder` → `feats2` ([B, 256, H, W] для DeepLabV3Plus)
- `F.interpolate(feats2, size=feats1.shape[-2:])` — выравниваем пространственно
- `cat([feats1, feats2], dim=1)` → **метаголовка** (несколько conv-слоёв)

Базовые модели заморожены (`requires_grad=False`), обучается только `meta_head`.

In [ ]:
def _build_smp_model(cfg: dict, weights_path: str | None = None) -> nn.Module:
    """Строит smp-модель и опционально загружает веса."""
    model_cls = getattr(smp, cfg["model_name"])
    model = model_cls(
        encoder_name    = cfg["encoder_name"],
        encoder_weights = cfg["encoder_weights"] if weights_path is None else None,
        in_channels     = 3,
        classes         = 1,
        activation      = None,
    )
    if weights_path is not None:
        ckpt = torch.load(weights_path, map_location="cpu")
        state = ckpt.get("model_state_dict", ckpt)

        # Убираем префикс module. (если обучалось с DataParallel/SWA)
        state = {k.replace("module.", ""): v for k, v in state.items()}

        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"  Loaded {weights_path}")
        if missing:     print(f"  Missing keys  : {missing[:5]}...")
        if unexpected:  print(f"  Unexpected    : {unexpected[:5]}...")
    return model


def _freeze(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = False
    module.eval()


class FeatureStackModel(nn.Module):
    def __init__(self, model1, model2):
        super().__init__()
        self.model1 = model1
        self.model2 = model2

        # Замораживаем базовые модели
        for param in self.model1.parameters():
            param.requires_grad = False
        for param in self.model2.parameters():
            param.requires_grad = False

        # "Голова" теперь принимает всего 2 канала (логиты первой и второй модели)
        # Используем 1x1 свертку для поиска оптимальных весов
        self.meta_head = nn.Conv2d(in_channels=2, out_channels=1, kernel_size=1)

    def forward(self, x1, x2):
        # x1 и x2 - входные тензоры разного размера (если нужно)

        with torch.no_grad():
            # Получаем финальные логиты (маски) от обеих моделей
            logits1 = self.model1(x1) # Ожидаем [B, 1, H1, W1]
            logits2 = self.model2(x2) # Ожидаем [B, 1, H2, W2]

            # Приводим вторую маску к размеру первой, если они отличаются
            if logits1.shape[-2:] != logits2.shape[-2:]:
                logits2 = F.interpolate(logits2, size=logits1.shape[-2:],
                                      mode='bilinear', align_corners=False)

        # Соединяем логиты в один тензор [B, 2, H, W]
        combined_logits = torch.cat([logits1, logits2], dim=1)

        # Мета-голова обучается их смешивать
        out = self.meta_head(combined_logits)
        return out
    # """
    # Стекинговая модель на основе декодерных признаков двух замороженных моделей.

    # Параметры:
    #     model1, model2  : smp-модели с атрибутами .encoder / .decoder
    #     meta_in_channels: суммарное кол-во каналов (feat1 + feat2)
    # """

    # def __init__(self, model1: nn.Module, model2: nn.Module, meta_in_channels: int):
    #     super().__init__()

    #     self.model1 = model1
    #     self.model2 = model2
    #     _freeze(self.model1)
    #     _freeze(self.model2)

    #     # Лёгкая метаголовка: conv → BN → ReLU → conv → BN → ReLU → 1x1
    #     self.meta_head = nn.Sequential(
    #         nn.Conv2d(meta_in_channels, 128, kernel_size=3, padding=1, bias=False),
    #         nn.BatchNorm2d(128),
    #         nn.ReLU(inplace=True),
    #         nn.Conv2d(128, 64, kernel_size=3, padding=1, bias=False),
    #         nn.BatchNorm2d(64),
    #         nn.ReLU(inplace=True),
    #         nn.Conv2d(64, 1, kernel_size=1),  # логиты
    #     )

    # def _extract_features(self, model: nn.Module, x: torch.Tensor) -> torch.Tensor:
    #     """
    #     Прогоняет x через encoder → decoder, возвращает признаки
    #     ДО segmentation_head (т.е. сырые декодерные карты признаков).
    #     """
    #     features = model.encoder(x)
    #     decoder_output = model.decoder(features)
    #     return decoder_output  # [B, C, H, W]

    # def forward(self, img1: torch.Tensor, img2: torch.Tensor) -> torch.Tensor:
    #     """
    #     Args:
    #         img1: [B, 3, H, W] — препроцессинг под encoder1
    #         img2: [B, 3, H, W] — препроцессинг под encoder2
    #     Returns:
    #         logits: [B, 1, H, W]
    #     """
    #     with torch.no_grad():  # базовые модели заморожены
    #         feats1 = self._extract_features(self.model1, img1)  # [B, C1, H, W]
    #         feats2 = self._extract_features(self.model2, img2)  # [B, C2, H, W]

    #     # Если пространственные размеры отличаются — выравниваем к feats1
    #     if feats1.shape[-2:] != feats2.shape[-2:]:
    #         feats2 = F.interpolate(
    #             feats2,
    #             size=feats1.shape[-2:],
    #             mode="bilinear",
    #             align_corners=False,
    #         )

    #     combined = torch.cat([feats1, feats2], dim=1)  # [B, C1+C2, H, W]
    #     logits = self.meta_head(combined)               # [B, 1, H, W]
        return logits

    def train(self, mode: bool = True):
        """Держим базовые модели всегда в eval(), обучаем только meta_head."""
        super().train(mode)
        _freeze(self.model1)
        _freeze(self.model2)
        return self

## 11. Определяем размерность декодерных признаков

In [ ]:
print("Building base models and detecting decoder output channels...")

# Строим модели с предобученными весами
_m1 = _build_smp_model(MODEL1_CONFIG, weights_path=str(ckpt1_local)).to(DEVICE)
_m2 = _build_smp_model(MODEL2_CONFIG, weights_path=str(ckpt2_local)).to(DEVICE)
_m1.eval()
_m2.eval()

with torch.no_grad():
    _dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
    _enc1 = _m1.encoder(_dummy)
    _f1   = _m1.decoder(_enc1)

    _enc2 = _m2.encoder(_dummy)
    _f2   = _m2.decoder(_enc2)

C1 = _f1.shape[1]
C2 = _f2.shape[1]
META_IN = C1 + C2

print(f"Model1 decoder output: {tuple(_f1.shape)}  →  {C1} channels")
print(f"Model2 decoder output: {tuple(_f2.shape)}  →  {C2} channels")
print(f"Meta-head input channels: {META_IN}")

del _dummy, _f1, _f2
torch.cuda.empty_cache()

## 12. Собираем стекинговую модель

In [ ]:
stacked_model = FeatureStackModel(_m1, _m2, meta_in_channels=META_IN).to(DEVICE)

# Проверяем: обучаемые параметры только в meta_head
total_params     = sum(p.numel() for p in stacked_model.parameters())
trainable_params = sum(p.numel() for p in stacked_model.parameters() if p.requires_grad)

print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}  ({100*trainable_params/total_params:.2f}%)")

## 13. Data split

In [ ]:
# Получаем полный список масок для split
_all_masks = sorted(MASKS_DIR.glob("*.png"))
indices = list(range(len(_all_masks)))

train_idx, val_idx = train_test_split(indices, test_size=VAL_RATIO, random_state=SEED)
print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")

_dataset_kwargs = dict(
    images_dir   = IMAGES_DIR,
    masks_dir    = MASKS_DIR,
    img_size     = IMG_SIZE,
    enc1_name    = MODEL1_CONFIG["encoder_name"],
    enc1_weights = MODEL1_CONFIG["encoder_weights"],
    enc2_name    = MODEL2_CONFIG["encoder_name"],
    enc2_weights = MODEL2_CONFIG["encoder_weights"],
)

train_dataset = torch.utils.data.Subset(
    StackedDataset(**_dataset_kwargs, transforms=get_train_transforms(IMG_SIZE)),
    train_idx,
)

val_dataset = torch.utils.data.Subset(
    StackedDataset(**_dataset_kwargs, transforms=get_val_transforms(IMG_SIZE)),
    val_idx,
)

train_loader = DataLoader(
    train_dataset,
    batch_size        = BATCH_SIZE,
    shuffle           = True,
    num_workers       = NUM_WORKERS,
    pin_memory        = True,
    prefetch_factor   = PREFETCH_FACTOR,
    persistent_workers= PERSISTENT_WORKERS,
)

val_loader = DataLoader(
    val_dataset,
    batch_size        = BATCH_SIZE,
    shuffle           = False,
    num_workers       = NUM_WORKERS,
    pin_memory        = True,
    prefetch_factor   = PREFETCH_FACTOR,
    persistent_workers= PERSISTENT_WORKERS,
)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 14. Optimizer & Scheduler

In [ ]:
import torch

# Оптимизируем только параметры meta_head
optimizer = torch.optim.AdamW(
    stacked_model.meta_head.parameters(),
    lr           = LR,
    weight_decay = WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS
)

scaler = torch.cuda.amp.GradScaler("cuda")
print("Optimizer ready")

## 15. Train / Val loops

In [ ]:
# def train_one_epoch(model, loader, optimizer, loss_fn, device):
#     model.train()
#     running_loss = running_dice = running_iou = 0.0

#     pbar = tqdm(loader, desc="Train", leave=False, file=sys.stdout)
#     for img1, img2, masks in pbar:
#         img1  = img1.to(device, non_blocking=True)
#         img2  = img2.to(device, non_blocking=True)
#         masks = masks.to(device, non_blocking=True)

#         optimizer.zero_grad(set_to_none=True)

#         with torch.amp.autocast('cuda'):
#             logits = model(img1, img2)
#             loss   = loss_fn(logits, masks)

#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()

#         running_loss += loss.item()
#         running_dice += dice_score_from_logits(logits.detach(), masks)
#         running_iou  += iou_score_from_logits(logits.detach(), masks)

#         pbar.set_postfix(loss=f"{loss.item():.4f}")

#     n = len(loader)
#     return {"loss": running_loss / n, "dice": running_dice / n, "iou": running_iou / n}

def dice_score(logits, masks, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    masks = masks.float()
    intersection = (preds * masks).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3))
    dice = (2 * intersection + 1e-6) / (union + 1e-6)
    return dice.mean()

def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, total_dice = 0.0, 0.0

    pbar = tqdm(loader, desc="Train", leave=False, file=sys.stdout)
    for img1, img2, masks in pbar:
        img1  = img1.to(device, non_blocking=True)
        img2  = img2.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(img1, img2)
            loss   = loss_fn(logits, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_dice += dice_score(logits, masks).item()

    n = len(loader)
    return {"loss": total_loss / n, "dice": total_dice / n}


@torch.no_grad()
def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()
    running_loss = running_dice = running_iou = 0.0

    for img1, img2, masks in tqdm(loader, desc="Val", leave=False, file=sys.stdout):
        img1  = img1.to(device, non_blocking=True)
        img2  = img2.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(img1, img2)
            loss   = loss_fn(logits, masks)

        running_loss += loss.item()
        running_dice += dice_score_from_logits(logits, masks)
        running_iou  += iou_score_from_logits(logits, masks)

    n = len(loader)
    return {"loss": running_loss / n, "dice": running_dice / n, "iou": running_iou / n}

## 16. Training loop

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(DEVICE)

In [ ]:
import subprocess
import numpy

def backup_best_to_drive(local_path: Path):
    remote = f"{GDRIVE_REMOTE}:{GDRIVE_SAVE_PATH}/stacked/best.pth"
    r = subprocess.run(
        ["rclone", "copyto", str(local_path), remote],
        capture_output=True, text=True
    )
    if r.returncode == 0:
        print(f"  ✓ Backed up → {remote}")
    else:
        print(f"  ✗ Drive backup failed: {r.stderr.strip()}")


best_val_dice = -1.0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_m = train_one_epoch(stacked_model, train_loader, optimizer, combined_loss, DEVICE)
    val_m   = validate_one_epoch(stacked_model, val_loader, combined_loss, DEVICE)
    scheduler.step()

    row = {
        "epoch"      : epoch,
        "lr"         : optimizer.param_groups[0]["lr"],
        "train_loss" : train_m["loss"],
        "train_dice" : train_m["dice"],
        "val_loss"   : val_m["loss"],
        "val_dice"   : val_m["dice"],
        "val_iou"    : val_m["iou"],
    }
    history.append(row)

    is_best = val_m["dice"] > best_val_dice
    if is_best:
        best_val_dice = val_m["dice"]
        ckpt_path = SAVE_DIR / "best.pth"
        torch.save(
            {
                "epoch"           : epoch,
                "model_state_dict": stacked_model.meta_head.state_dict(),
                "val_dice"        : best_val_dice,
                "model1_config"   : MODEL1_CONFIG,
                "model2_config"   : MODEL2_CONFIG,
                "meta_in_channels": META_IN,
                "img_size"        : IMG_SIZE,
                "threshold"       : THRESHOLD,
            },
            ckpt_path,
        )
        backup_best_to_drive(ckpt_path)

    flag = "★" if is_best else " "
    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS} {flag} "
        f"| lr={row['lr']:.1e} "
        f"| train loss={train_m['loss']:.4f} dice={train_m['dice']:.4f} "
        f"| val   loss={val_m['loss']:.4f}  dice={val_m['dice']:.4f}  iou={val_m['iou']:.4f}"
    )

print(f"\nBest val Dice: {best_val_dice:.4f}")

## 17. Loss / Dice curves

In [ ]:
import matplotlib.pyplot as plt

epochs = [r["epoch"] for r in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs, [r["train_loss"] for r in history], label="train")
axes[0].plot(epochs, [r["val_loss"]   for r in history], label="val")
axes[0].set_title("Loss")
axes[0].legend()

axes[1].plot(epochs, [r["train_dice"] for r in history], label="train")
axes[1].plot(epochs, [r["val_dice"]   for r in history], label="val")
axes[1].set_title("Dice")
axes[1].legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / "curves.png", dpi=100)
plt.show()

## 18. Submission

Загружаем лучший чекпоинт, прогоняем TTA (flip), пишем CSV.

In [ ]:
import json
import pandas as pd

TEST_IMAGES_DIR = Path("/kaggle/input/competitions/dl-lab-3-product-segmentation/test_images")
OUTPUT_CSV      = "/kaggle/working/submission.csv"
IMG_EXTS        = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# ── Загружаем лучший чекпоинт ──────────────────────────────
ckpt = torch.load(SAVE_DIR / "best.pth", map_location=DEVICE)

# Пересобираем модель (на случай если переменные потеряны)
m1 = _build_smp_model(ckpt["model1_config"], weights_path=str(ckpt1_local)).to(DEVICE)
m2 = _build_smp_model(ckpt["model2_config"], weights_path=str(ckpt2_local)).to(DEVICE)
inf_model = FeatureStackModel(m1, m2, meta_in_channels=ckpt["meta_in_channels"]).to(DEVICE)
inf_model.meta_head.load_state_dict(ckpt["model_state_dict"])
inf_model.eval()

inf_img_size = ckpt["img_size"]
inf_threshold = ckpt["threshold"]

preprocess1 = get_preprocessing_fn(
    ckpt["model1_config"]["encoder_name"],
    pretrained=ckpt["model1_config"]["encoder_weights"],
)
preprocess2 = get_preprocessing_fn(
    ckpt["model2_config"]["encoder_name"],
    pretrained=ckpt["model2_config"]["encoder_weights"],
)

print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_dice={ckpt['val_dice']:.4f})")


# ── Хелперы для инференса ──────────────────────────────────
def cv2_imread_unicode(path: Path, flags=cv2.IMREAD_COLOR):
    data = np.fromfile(str(path), dtype=np.uint8)
    return cv2.imdecode(data, flags) if data.size > 0 else None


def preprocess_for_inference(img_rgb: np.ndarray, size: int, preprocess_fn):
    """Ресайз + encoder-specific preprocessing → CHW float32 tensor."""
    img = cv2.resize(img_rgb, (size, size))
    img = img.astype(np.float32)
    if preprocess_fn is not None:
        img = preprocess_fn(img)
    else:
        img = img / 255.0
    return torch.from_numpy(img.transpose(2, 0, 1)).float()


def mask_to_rle(mask: np.ndarray) -> str:
    """Конвертирует бинарную маску (H×W, bool/uint8) в RLE-строку."""
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)


# ── TTA: оригинал + горизонтальный флип ───────────────────
@torch.no_grad()
def predict_with_tta(img_rgb: np.ndarray) -> np.ndarray:
    """Возвращает усреднённую вероятностную карту [H, W]."""
    H, W = img_rgb.shape[:2]

    def _infer(img):
        t1 = preprocess_for_inference(img, inf_img_size, preprocess1).unsqueeze(0).to(DEVICE)
        t2 = preprocess_for_inference(img, inf_img_size, preprocess2).unsqueeze(0).to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = inf_model(t1, t2)           # [1, 1, H, W]
        prob = torch.sigmoid(logits)[0, 0]       # [H, W]
        return prob.cpu().numpy()

    prob_orig = _infer(img_rgb)
    prob_flip = _infer(img_rgb[:, ::-1].copy())[:, ::-1].copy()
    prob_avg  = (prob_orig + prob_flip) / 2.0

    return cv2.resize(prob_avg, (W, H), interpolation=cv2.INTER_LINEAR)


# ── Основной цикл инференса ────────────────────────────────
image_paths = sorted([p for p in TEST_IMAGES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS])
print(f"Found {len(image_paths)} test images")

rows = []
for i, img_path in enumerate(image_paths, 1):
    img_bgr = cv2_imread_unicode(img_path)
    if img_bgr is None:
        print(f"[skip] {img_path}")
        continue

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    prob    = predict_with_tta(img_rgb)
    mask    = (prob > inf_threshold).astype(np.uint8)
    rle     = mask_to_rle(mask)

    rows.append({"image_id": img_path.stem, "segmentation": rle})

    if i % 50 == 0:
        print(f"  {i}/{len(image_paths)} processed")

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nSubmission saved → {OUTPUT_CSV}")
print(df.head())